In [2]:
import torch 
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import os
import tensorflow as tf


def load_model(model_path):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    return interpreter, input_details, output_details

val_transform = transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def preprocess_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img = val_transform(img)
    img = np.array(img, dtype=np.float32) 
    img = np.transpose(img, (1, 2, 0))
    img = np.expand_dims(img, axis=0)
    return img.astype(np.float32)

def extract_embedding(image_path):
    img = preprocess_image(image_path)
    
    interpreter.set_tensor(input_details[0]['index'], img)
    interpreter.invoke()
    embedding = interpreter.get_tensor(output_details[0]['index'])
    embedding = np.array(embedding).reshape(-1)
    return embedding.tolist()



import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

MODEL_PATH = "ver2_tf_cbir_256\\ver2_model_float16.tflite"
VECTOR_DATESET_PATH = "ver2_embedding_dataset.json"
TEST_IMAGE_PATH = input("Enter the path of the test image: ")

interpreter, input_details, output_details = load_model(MODEL_PATH)
with open(VECTOR_DATESET_PATH, 'r', encoding="utf-8") as f:
    vector_dataset = json.load(f)

# Lấy danh sách path và vectors
dataset_image_paths = [item["id"] for item in vector_dataset]
dataset_vectors = np.array([item["vector"] for item in vector_dataset])


output_vector = np.array(extract_embedding(TEST_IMAGE_PATH)).reshape(1, -1)

# Tính cosine similarity
similarity = cosine_similarity(output_vector, dataset_vectors)
print("Similarity:", similarity)

# top-k ảnh giống nhất
top_k = 10
indices = np.argsort(similarity[0])[::-1][:top_k]
for idx in indices:
    print(dataset_image_paths[idx], similarity[0][idx])


d:\THAI\PROJECT\CBIR_ON_EDGE_DEVICE\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Similarity: [[0.45398744 0.18042588 0.54348487 ... 0.088797   0.06831394 0.31224266]]
..\..\DATA_NEW_merged\microphone_80.jpg 0.7976921354980885
..\..\DATA_NEW_merged\Speaker_25.jpg 0.7675249801989692
..\..\DATA_NEW_merged\keys_objects_69.jpg 0.7653590987245782
..\..\DATA_NEW_merged\ceiling_fan_20.jpg 0.7650668783064491
..\..\DATA_NEW_merged\Speaker_12.jpg 0.7640330896505458
..\..\DATA_NEW_merged\Speaker_41.jpg 0.7607723647021116
..\..\DATA_NEW_merged\hard_disk_140.jpg 0.7583181687699814
..\..\DATA_NEW_merged\keys_objects_33.jpg 0.7548930356810595
..\..\DATA_NEW_merged\projector_61.jpg 0.7520966920425393
..\..\DATA_NEW_merged\Speaker_46.jpg 0.7517364433297526


In [3]:
print(interpreter.get_input_details())

[{'name': 'input', 'index': 0, 'shape': array([  1, 224, 224,   3]), 'shape_signature': array([  1, 224, 224,   3]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
